**DATA AQUISITION OF ALL 5 CITIES FOR FULLLOAD THROUGH API**

In [0]:
import requests
import time

RAW_DIR = "dbfs:/Volumes/workspace/default/products"

variables = [
    "temperature_2m", "relative_humidity_2m", "dew_point_2m", "apparent_temperature",
    "pressure_msl", "surface_pressure", "weather_code", "vapour_pressure_deficit",
    "precipitation", "rain", "snowfall", "snow_depth",
    "wind_speed_10m", "wind_speed_100m", "wind_direction_10m", "wind_direction_100m", "wind_gusts_10m",
    "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high",
    "shortwave_radiation", "direct_radiation", "diffuse_radiation",
    "direct_normal_irradiance", "global_tilted_irradiance", "terrestrial_radiation",
    "shortwave_radiation_instant", "direct_radiation_instant", "diffuse_radiation_instant",
    "direct_normal_irradiance_instant", "global_tilted_irradiance_instant", "terrestrial_radiation_instant",
    "soil_temperature_0_to_7cm", "soil_temperature_7_to_28cm",
    "soil_temperature_28_to_100cm", "soil_temperature_100_to_255cm",
    "soil_moisture_0_to_7cm", "soil_moisture_7_to_28cm",
    "soil_moisture_28_to_100cm", "soil_moisture_100_to_255cm",
    "et0_fao_evapotranspiration"
]
variables_str = ",".join(variables)

chunks = [
    ("2000-01-01", "2003-12-31"),
    ("2004-01-01", "2007-12-31"),
    ("2008-01-01", "2011-12-31"),
    ("2012-01-01", "2015-12-31"),
    ("2016-01-01", "2019-12-31"),
    ("2020-01-01", "2026-09-30"),
]

# ═══════════════════════════════════════════
CITY_NAME = "Lahore"
CITY_LAT  = 31.5204
CITY_LON  = 74.3587
# ═══════════════════════════════════════════

# Run 1 → Lahore    | 31.5204 | 74.3587
# Run 2 → Karachi   | 24.8607 | 67.0011
# Run 3 → Islamabad | 33.6844 | 73.0479
# Run 4 → Peshawar  | 34.0151 | 71.5249
# Run 5 → Quetta    | 30.1798 | 66.9750

raw_payloads = []
print(f"{'='*50}")
print(f"  Fetching: {CITY_NAME}")
print(f"  Chunks: {len(chunks)} | Variables: {len(variables)}")
print(f"{'='*50}")

for i, (chunk_start, chunk_end) in enumerate(chunks, 1):
    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={CITY_LAT}&longitude={CITY_LON}"
        f"&start_date={chunk_start}&end_date={chunk_end}"
        f"&hourly={variables_str}"
    )

    print(f"  [{i}/6] {chunk_start} → {chunk_end}...", end=" ", flush=True)

    for attempt in range(1, 6):
        try:
            response = requests.get(url, timeout=120)
            if response.status_code == 200:
                raw_payloads.append(response.text)
                print("✓ Success")
                break
            elif response.status_code == 429:
                wait = 90 * attempt
                print(f"\n    Rate limited. Waiting {wait}s...", end=" ")
                time.sleep(wait)
            else:
                print(f"\n    Status {response.status_code}. Waiting 30s...")
                time.sleep(30)
        except Exception as e:
            print(f"\n    Error: {e}. Waiting 30s...")
            time.sleep(30)

    time.sleep(10)  

print(f"\nSaving {CITY_NAME}...")

SAVE_PATH = f"{RAW_DIR}/{CITY_NAME.lower()}_raw.json"
combined  = "[\n" + ",\n".join(raw_payloads) + "\n]"
temp      = f"{RAW_DIR}/temp_{CITY_NAME.lower()}"

spark.createDataFrame([(combined,)], ["value"]) \
    .coalesce(1).write.mode("overwrite").text(temp)

for f in dbutils.fs.ls(temp):
    if f.name.startswith("part-"):
        dbutils.fs.mv(f.path, SAVE_PATH)
    elif f.name.startswith("_"):
        dbutils.fs.rm(f.path)

dbutils.fs.rm(temp, True)

size = dbutils.fs.ls(SAVE_PATH)[0].size / 1024 / 1024
print(f"\n{'='*50}")
print(f"  {CITY_NAME} SAVED!")
print(f"  Path : {SAVE_PATH}")
print(f"  Size : {size:.1f} MB")
print(f"  Chunks saved: {len(raw_payloads)}/6")
print(f"{'='*50}")
print(f"\n  NEXT RUN → Change CITY_NAME to next city!")

In [0]:

import requests
import time

RAW_DIR = "dbfs:/Volumes/workspace/default/products"

variables = [
    "temperature_2m", "relative_humidity_2m", "dew_point_2m", "apparent_temperature",
    "pressure_msl", "surface_pressure", "weather_code", "vapour_pressure_deficit",
    "precipitation", "rain", "snowfall", "snow_depth",
    "wind_speed_10m", "wind_speed_100m", "wind_direction_10m", "wind_direction_100m", "wind_gusts_10m",
    "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high",
    "shortwave_radiation", "direct_radiation", "diffuse_radiation",
    "direct_normal_irradiance", "global_tilted_irradiance", "terrestrial_radiation",
    "shortwave_radiation_instant", "direct_radiation_instant", "diffuse_radiation_instant",
    "direct_normal_irradiance_instant", "global_tilted_irradiance_instant", "terrestrial_radiation_instant",
    "soil_temperature_0_to_7cm", "soil_temperature_7_to_28cm",
    "soil_temperature_28_to_100cm", "soil_temperature_100_to_255cm",
    "soil_moisture_0_to_7cm", "soil_moisture_7_to_28cm",
    "soil_moisture_28_to_100cm", "soil_moisture_100_to_255cm",
    "et0_fao_evapotranspiration"
]
variables_str = ",".join(variables)

chunks = [
    ("2000-01-01", "2003-12-31"),
    ("2004-01-01", "2007-12-31"),
    ("2008-01-01", "2011-12-31"),
    ("2012-01-01", "2015-12-31"),
    ("2016-01-01", "2019-12-31"),
    ("2020-01-01", "2026-09-30"),
]

# ═══════════════════════════════════════════
CITY_NAME = "Quetta"
CITY_LAT  = 30.1798
CITY_LON  =  66.9750
# ═══════════════════════════════════════════

# Run 1 → Lahore    | 31.5204 | 74.3587
# Run 2 → Karachi   | 24.8607 | 67.0011
# Run 3 → Islamabad | 33.6844 | 73.0479
# Run 4 → Peshawar  | 34.0151 | 71.5249
# Run 5 → Quetta    | 30.1798 | 66.9750

raw_payloads = []
print(f"{'='*50}")
print(f"  Fetching: {CITY_NAME}")
print(f"  Chunks: {len(chunks)} | Variables: {len(variables)}")
print(f"{'='*50}")

for i, (chunk_start, chunk_end) in enumerate(chunks, 1):
    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={CITY_LAT}&longitude={CITY_LON}"
        f"&start_date={chunk_start}&end_date={chunk_end}"
        f"&hourly={variables_str}"
    )

    print(f"  [{i}/6] {chunk_start} → {chunk_end}...", end=" ", flush=True)

    for attempt in range(1, 6):
        try:
            response = requests.get(url, timeout=120)
            if response.status_code == 200:
                raw_payloads.append(response.text)
                print("✓ Success")
                break
            elif response.status_code == 429:
                wait = 90 * attempt
                print(f"\n    Rate limited. Waiting {wait}s...", end=" ")
                time.sleep(wait)
            else:
                print(f"\n    Status {response.status_code}. Waiting 30s...")
                time.sleep(30)
        except Exception as e:
            print(f"\n    Error: {e}. Waiting 30s...")
            time.sleep(30)

    time.sleep(10)  

print(f"\nSaving {CITY_NAME}...")

SAVE_PATH = f"{RAW_DIR}/{CITY_NAME.lower()}_raw.json"
combined  = "[\n" + ",\n".join(raw_payloads) + "\n]"
temp      = f"{RAW_DIR}/temp_{CITY_NAME.lower()}"

spark.createDataFrame([(combined,)], ["value"]) \
    .coalesce(1).write.mode("overwrite").text(temp)

for f in dbutils.fs.ls(temp):
    if f.name.startswith("part-"):
        dbutils.fs.mv(f.path, SAVE_PATH)
    elif f.name.startswith("_"):
        dbutils.fs.rm(f.path)

dbutils.fs.rm(temp, True)

size = dbutils.fs.ls(SAVE_PATH)[0].size / 1024 / 1024
print(f"\n{'='*50}")
print(f"  {CITY_NAME} SAVED!")
print(f"  Path : {SAVE_PATH}")
print(f"  Size : {size:.1f} MB")
print(f"  Chunks saved: {len(raw_payloads)}/6")
print(f"{'='*50}")
print(f"\n  NEXT RUN → Change CITY_NAME to next city!")

In [0]:
#islamabad and peshawar 
import requests
import time

RAW_DIR = "dbfs:/Volumes/workspace/default/products"

variables = [
    "temperature_2m", "relative_humidity_2m", "dew_point_2m", "apparent_temperature",
    "pressure_msl", "surface_pressure", "weather_code", "vapour_pressure_deficit",
    "precipitation", "rain", "snowfall", "snow_depth",
    "wind_speed_10m", "wind_speed_100m", "wind_direction_10m", "wind_direction_100m", "wind_gusts_10m",
    "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high",
    "shortwave_radiation", "direct_radiation", "diffuse_radiation",
    "direct_normal_irradiance", "global_tilted_irradiance", "terrestrial_radiation",
    "shortwave_radiation_instant", "direct_radiation_instant", "diffuse_radiation_instant",
    "direct_normal_irradiance_instant", "global_tilted_irradiance_instant", "terrestrial_radiation_instant",
    "soil_temperature_0_to_7cm", "soil_temperature_7_to_28cm",
    "soil_temperature_28_to_100cm", "soil_temperature_100_to_255cm",
    "soil_moisture_0_to_7cm", "soil_moisture_7_to_28cm",
    "soil_moisture_28_to_100cm", "soil_moisture_100_to_255cm",
    "et0_fao_evapotranspiration"
]
variables_str = ",".join(variables)

chunks = [
    ("2000-01-01", "2003-12-31"),
    ("2004-01-01", "2007-12-31"),
    ("2008-01-01", "2011-12-31"),
    ("2012-01-01", "2015-12-31"),
    ("2016-01-01", "2019-12-31"),
    ("2020-01-01", "2026-09-30"),
]

# ═══════════════════════════════════════════
CITY_NAME = "Islamabad"
CITY_LAT  = 33.6844
CITY_LON  = 73.0479
# ═══════════════════════════════════════════

# Run 1 → Lahore    | 31.5204 | 74.3587
# Run 2 → Karachi   | 24.8607 | 67.0011
# Run 3 → Islamabad | 33.6844 | 73.0479
# Run 4 → Peshawar  | 34.0151 | 71.5249
# Run 5 → Quetta    | 30.1798 | 66.9750

raw_payloads = []
print(f"{'='*50}")
print(f"  Fetching: {CITY_NAME}")
print(f"  Chunks: {len(chunks)} | Variables: {len(variables)}")
print(f"{'='*50}")

for i, (chunk_start, chunk_end) in enumerate(chunks, 1):
    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={CITY_LAT}&longitude={CITY_LON}"
        f"&start_date={chunk_start}&end_date={chunk_end}"
        f"&hourly={variables_str}"
    )

    print(f"  [{i}/6] {chunk_start} → {chunk_end}...", end=" ", flush=True)

    for attempt in range(1, 6):
        try:
            response = requests.get(url, timeout=120)
            if response.status_code == 200:
                raw_payloads.append(response.text)
                print("✓ Success")
                break
            elif response.status_code == 429:
                wait = 90 * attempt
                print(f"\n    Rate limited. Waiting {wait}s...", end=" ")
                time.sleep(wait)
            else:
                print(f"\n    Status {response.status_code}. Waiting 30s...")
                time.sleep(30)
        except Exception as e:
            print(f"\n    Error: {e}. Waiting 30s...")
            time.sleep(30)

    time.sleep(10)  

print(f"\nSaving {CITY_NAME}...")

SAVE_PATH = f"{RAW_DIR}/{CITY_NAME.lower()}_raw.json"
combined  = "[\n" + ",\n".join(raw_payloads) + "\n]"
temp      = f"{RAW_DIR}/temp_{CITY_NAME.lower()}"

spark.createDataFrame([(combined,)], ["value"]) \
    .coalesce(1).write.mode("overwrite").text(temp)

for f in dbutils.fs.ls(temp):
    if f.name.startswith("part-"):
        dbutils.fs.mv(f.path, SAVE_PATH)
    elif f.name.startswith("_"):
        dbutils.fs.rm(f.path)

dbutils.fs.rm(temp, True)

size = dbutils.fs.ls(SAVE_PATH)[0].size / 1024 / 1024
print(f"\n{'='*50}")
print(f"  {CITY_NAME} SAVED!")
print(f"  Path : {SAVE_PATH}")
print(f"  Size : {size:.1f} MB")
print(f"  Chunks saved: {len(raw_payloads)}/6")
print(f"{'='*50}")
print(f"\n  NEXT RUN → Change CITY_NAME to next city!")

In [0]:
import requests
import time

RAW_DIR = "dbfs:/Volumes/workspace/default/products"

variables = [
    "temperature_2m", "relative_humidity_2m", "dew_point_2m", "apparent_temperature",
    "pressure_msl", "surface_pressure", "weather_code", "vapour_pressure_deficit",
    "precipitation", "rain", "snowfall", "snow_depth",
    "wind_speed_10m", "wind_speed_100m", "wind_direction_10m", "wind_direction_100m", "wind_gusts_10m",
    "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high",
    "shortwave_radiation", "direct_radiation", "diffuse_radiation",
    "direct_normal_irradiance", "global_tilted_irradiance", "terrestrial_radiation",
    "shortwave_radiation_instant", "direct_radiation_instant", "diffuse_radiation_instant",
    "direct_normal_irradiance_instant", "global_tilted_irradiance_instant", "terrestrial_radiation_instant",
    "soil_temperature_0_to_7cm", "soil_temperature_7_to_28cm",
    "soil_temperature_28_to_100cm", "soil_temperature_100_to_255cm",
    "soil_moisture_0_to_7cm", "soil_moisture_7_to_28cm",
    "soil_moisture_28_to_100cm", "soil_moisture_100_to_255cm",
    "et0_fao_evapotranspiration"
]
variables_str = ",".join(variables)

chunks = [
    ("2000-01-01", "2003-12-31"),
    ("2004-01-01", "2007-12-31"),
    ("2008-01-01", "2011-12-31"),
    ("2012-01-01", "2015-12-31"),
    ("2016-01-01", "2019-12-31"),
    ("2020-01-01", "2026-09-30"),
]

# ═══════════════════════════════════════════
CITY_NAME = "Peshawar"
CITY_LAT  = 34.0151
CITY_LON  = 71.5249
# ═══════════════════════════════════════════

# Run 1 → Lahore    | 31.5204 | 74.3587
# Run 2 → Karachi   | 24.8607 | 67.0011
# Run 3 → Islamabad | 33.6844 | 73.0479
# Run 4 → Peshawar  | 34.0151 | 71.5249
# Run 5 → Quetta    | 30.1798 | 66.9750

raw_payloads = []
print(f"{'='*50}")
print(f"  Fetching: {CITY_NAME}")
print(f"  Chunks: {len(chunks)} | Variables: {len(variables)}")
print(f"{'='*50}")

for i, (chunk_start, chunk_end) in enumerate(chunks, 1):
    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={CITY_LAT}&longitude={CITY_LON}"
        f"&start_date={chunk_start}&end_date={chunk_end}"
        f"&hourly={variables_str}"
    )

    print(f"  [{i}/6] {chunk_start} → {chunk_end}...", end=" ", flush=True)

    for attempt in range(1, 6):
        try:
            response = requests.get(url, timeout=120)
            if response.status_code == 200:
                raw_payloads.append(response.text)
                print("✓ Success")
                break
            elif response.status_code == 429:
                wait = 90 * attempt
                print(f"\n    Rate limited. Waiting {wait}s...", end=" ")
                time.sleep(wait)
            else:
                print(f"\n    Status {response.status_code}. Waiting 30s...")
                time.sleep(30)
        except Exception as e:
            print(f"\n    Error: {e}. Waiting 30s...")
            time.sleep(30)

    time.sleep(10)  

print(f"\nSaving {CITY_NAME}...")

SAVE_PATH = f"{RAW_DIR}/{CITY_NAME.lower()}_raw.json"
combined  = "[\n" + ",\n".join(raw_payloads) + "\n]"
temp      = f"{RAW_DIR}/temp_{CITY_NAME.lower()}"

spark.createDataFrame([(combined,)], ["value"]) \
    .coalesce(1).write.mode("overwrite").text(temp)

for f in dbutils.fs.ls(temp):
    if f.name.startswith("part-"):
        dbutils.fs.mv(f.path, SAVE_PATH)
    elif f.name.startswith("_"):
        dbutils.fs.rm(f.path)

dbutils.fs.rm(temp, True)

size = dbutils.fs.ls(SAVE_PATH)[0].size / 1024 / 1024
print(f"\n{'='*50}")
print(f"  {CITY_NAME} SAVED!")
print(f"  Path : {SAVE_PATH}")
print(f"  Size : {size:.1f} MB")
print(f"  Chunks saved: {len(raw_payloads)}/6")
print(f"{'='*50}")
print(f"\n  NEXT RUN → Change CITY_NAME to next city!")

In [0]:
RAW_DIR = "dbfs:/Volumes/workspace/default/products"

files = dbutils.fs.ls(RAW_DIR)
print("Files saved so far:")
print("="*45)
for f in files:
    size = f.size / 1024 / 1024
    print(f"  {f.name:<40} {size:.1f} MB")

In [0]:
RAW_DIR    = "/Volumes/workspace/default/products"   # plain path for Python file I/O
cities     = ["lahore", "karachi", "islamabad", "peshawar", "quetta"]
FINAL_PATH = f"{RAW_DIR}/pakistan_weather_fullload.json"

print("Merging all cities...")

with open(FINAL_PATH, "w", encoding="utf-8") as out:
    out.write("[\n")
    for n, city in enumerate(cities):
        with open(f"{RAW_DIR}/{city}_raw.json", "r", encoding="utf-8") as f:
            text = f.read().strip()

        # each city file is "[ chunk, chunk, ... ]": drop only the outer brackets
        assert text.startswith("[") and text.endswith("]"), f"{city}: unexpected format"
        text = text[1:-1].strip()

        if n > 0:
            out.write(",\n")
        out.write(text)
        print(f"  {city}: {len(text) / 1024 / 1024:.1f} MB")
    out.write("\n]")

size = dbutils.fs.ls(f"dbfs:{FINAL_PATH}")[0].size / 1024 / 1024
print(f"\nALL CITIES MERGED!  {FINAL_PATH}  ({size:.1f} MB)")